# Train Nemotron 3 Nano with GRPO, Unsloth, and NeMo Gym

This notebook uses the current Unsloth release and a compatibility-tested training matrix. The pinned NVIDIA 4B checkpoint uses repository model code written for the pre-5.x generation API, so Transformers 4.56.2 is intentional rather than an unsafe model-specific monkey patch.

## 1. Start the real NeMo Gym 0.3 resource server

From the tutorial root, run this in a separate terminal:

```bash
uv run --project training ng_run "+config_paths=[resources_servers/langgraph_cli/configs/langgraph_cli.yaml]"
```

`ng_run` supplies Gym's head-server configuration and starts the verifier on `127.0.0.1:8000`. Do not launch this resource server with raw Uvicorn.

In [ ]:
import requests

NEMO_GYM_URL = "http://127.0.0.1:8000"
VERIFY_ENDPOINT = f"{NEMO_GYM_URL}/verify"
health = requests.get(f"{NEMO_GYM_URL}/", timeout=5)
health.raise_for_status()
print(health.json())

## 2. Verify the compatible training environment

Install it with `uv sync --project training`, then run `uv run --project training python install_ssm_kernels.py` on the CUDA training host. The recipe targets the 4B checkpoint; other Nemotron checkpoints can need different LoRA targets and memory tuning.

In [ ]:
from importlib.metadata import version

import torch
from datasets import Dataset
from unsloth import (
    FastLanguageModel,
)  # Import Unsloth before TRL so its compatibility patches are active.
from trl import GRPOConfig, GRPOTrainer

PACKAGE_NAMES = (
    "torch",
    "transformers",
    "trl",
    "unsloth",
    "nemo-gym",
    "causal-conv1d",
    "mamba-ssm",
)
for package in PACKAGE_NAMES:
    print(f"{package}: {version(package)}")
assert version("causal-conv1d") == "1.6.1"
assert version("mamba-ssm") == "2.3.1"
assert torch.cuda.is_available(), "GRPO training requires an NVIDIA CUDA GPU"
assert torch.cuda.is_bf16_supported(), "This recipe requires BF16-capable hardware"
print(torch.cuda.get_device_name(0))

## 3. Load Nemotron 3 Nano 4B

Unsloth 2026.6.9 contains native Nemotron-H handling and a general GRPO hidden-state fallback, so the legacy model-specific monkey patch is no longer needed. We use BF16 rather than bitsandbytes quantization for this hybrid Mamba-Transformer checkpoint. Reasoning is disabled consistently during training and inference because this task needs a short JSON tool call, not an intermediate trace.

In [ ]:
import os
from functools import partial

MODEL_NAME = os.getenv(
    "MODEL_NAME",
    "nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16",
)
MODEL_REVISION = os.getenv(
    "MODEL_REVISION",
    "dfaf35de3e30f1867dd8dbc38a7fc9fb52d3914f",
)
MAX_SEQ_LENGTH = 1024
LORA_RANK = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    trust_remote_code=True,
    revision=MODEL_REVISION,
    load_in_4bit=False,
    dtype=torch.bfloat16,
)
mamba_fast_path_available = None
if os.getenv("GRPO_REQUIRE_MAMBA_FAST_PATH", "0") == "1":
    import sys

    mamba_mixers = [
        module
        for module in model.modules()
        if module.__class__.__name__ == "NemotronHMamba2Mixer"
    ]
    assert mamba_mixers, "Loaded model did not contain a Nemotron-H Mamba mixer"
    mixer_implementation = sys.modules[mamba_mixers[0].__class__.__module__]
    required_kernel_symbols = (
        "selective_state_update",
        "mamba_chunk_scan_combined",
        "mamba_split_conv1d_scan_combined",
        "causal_conv1d_fn",
        "causal_conv1d_update",
    )
    missing_kernel_symbols = [
        name for name in required_kernel_symbols if not getattr(mixer_implementation, name, None)
    ]
    mamba_fast_path_available = bool(
        getattr(mixer_implementation, "is_fast_path_available", False)
    )
    assert mamba_fast_path_available and not missing_kernel_symbols, (
        "The loaded Nemotron-H implementation is missing fused kernels: "
        f"{missing_kernel_symbols}"
    )
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "up_proj",
        "down_proj",
        "in_proj",
        "out_proj",
    ],
    lora_alpha=2 * LORA_RANK,
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Nemotron 3 defaults to reasoning-on. TRL 0.24 predates the GRPOConfig
# chat_template_kwargs field, so bind reasoning-off at the tokenizer boundary.
# Every template call made by this single-GPU trainer now receives the same mode.
tokenizer.apply_chat_template = partial(
    tokenizer.apply_chat_template,
    enable_thinking=False,
)

The official 4B checkpoint currently declares custom Nemotron-H configuration/model classes and requires `trust_remote_code=True`. This notebook pins the reviewed NVIDIA repository revision rather than executing mutable `main`; review and update that revision deliberately when refreshing the model.

## 4. Load the Gym-native dataset

In [ ]:
import json
from pathlib import Path


def load_training_data(path: str) -> Dataset:
    rows = []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        record = json.loads(line)
        rows.append(
            {
                "prompt": record["responses_create_params"]["input"],
                "expected_output": record["expected_output"],
            }
        )
    return Dataset.from_list(rows)


TRAIN_FILE = os.getenv("GRPO_TRAIN_FILE", "data/langgraph_cli/train.jsonl")
VALIDATION_FILE = os.getenv(
    "GRPO_VALIDATION_FILE", "data/langgraph_cli/validation.jsonl"
)
train_dataset = load_training_data(TRAIN_FILE)
validation_dataset = load_training_data(VALIDATION_FILE)
print(len(train_dataset), len(validation_dataset))

## 5. Connect TRL's reward function to Gym

TRL passes dataset columns to custom reward functions by name. `expected_output` therefore travels with every sampled completion; there is no fragile lookup by prompt text. The helper returns `list[float]`, as TRL 0.24 requires.

In [ ]:
from tutorial_utils import build_verify_payload, create_nemo_gym_reward_function

example = train_dataset[0]
smoke_payload = build_verify_payload(
    [{"role": "assistant", "content": json.dumps(example["expected_output"])}],
    example["prompt"],
    example["expected_output"],
    "smoke",
)
smoke_response = requests.post(VERIFY_ENDPOINT, json=smoke_payload, timeout=30)
smoke_response.raise_for_status()
assert smoke_response.json()["reward"] == 1.0
smoke_response.json()

In [ ]:
reward_fn = create_nemo_gym_reward_function(VERIFY_ENDPOINT)

## 6. Configure and run GRPO

This small run validates the pipeline. Increase `max_steps` only after rewards and generated JSON look correct. vLLM is not needed for this single-GPU smoke recipe and is intentionally not installed.

In [ ]:
max_prompt_length = (
    max(
        len(
            tokenizer.apply_chat_template(
                row["prompt"],
                add_generation_prompt=True,
                tokenize=True,
                enable_thinking=False,
            )
        )
        for split in (train_dataset, validation_dataset)
        for row in split
    )
    + 16
)
max_completion_length = MAX_SEQ_LENGTH - max_prompt_length
assert max_completion_length >= 128
print(max_prompt_length, max_completion_length)

In [ ]:
GRPO_OUTPUT_DIR = os.getenv("GRPO_OUTPUT_DIR", "outputs/grpo_langgraph_cli")
GRPO_MAX_STEPS = int(os.getenv("GRPO_MAX_STEPS", "50"))
if GRPO_MAX_STEPS < 1:
    raise ValueError("GRPO_MAX_STEPS must be positive")

training_args = GRPOConfig(
    output_dir=GRPO_OUTPUT_DIR,
    temperature=0.6,
    top_p=0.95,
    num_generations=4,
    max_prompt_length=max_prompt_length,
    max_completion_length=max_completion_length,
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    max_steps=GRPO_MAX_STEPS,
    warmup_ratio=0.1,
    weight_decay=0.01,
    optim="adamw_torch_fused",
    lr_scheduler_type="cosine",
    epsilon_high=0.28,
    mask_truncated_completions=True,
    bf16=True,
    use_vllm=False,
    logging_steps=1,
    save_steps=50,
    report_to="none",
)
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
)

In [ ]:
import math

train_result = trainer.train()
training_loss = float(train_result.training_loss)
assert trainer.state.global_step == GRPO_MAX_STEPS
assert math.isfinite(training_loss)

run_evaluation = os.getenv("GRPO_RUN_EVAL", "1") == "1"
validation_metrics = trainer.evaluate() if run_evaluation else {}
numeric_validation_metrics = {}
for key, value in validation_metrics.items():
    try:
        numeric_value = float(value)
    except (TypeError, ValueError):
        continue
    assert math.isfinite(numeric_value), f"Non-finite evaluation metric: {key}"
    numeric_validation_metrics[key] = numeric_value

held_out_predictions = []
if run_evaluation:
    FastLanguageModel.for_inference(model)
    model_device = next(model.parameters()).device
    for index, row in enumerate(validation_dataset):
        prompt_ids = tokenizer.apply_chat_template(
            row["prompt"],
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            enable_thinking=False,
        ).to(model_device)
        with torch.inference_mode():
            generated_ids = model.generate(
                input_ids=prompt_ids,
                max_new_tokens=min(256, max_completion_length),
                do_sample=False,
                use_cache=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        completion = tokenizer.decode(
            generated_ids[0, prompt_ids.shape[-1] :], skip_special_tokens=True
        ).strip()
        held_out_reward = reward_fn(
            completions=[completion],
            prompts=[row["prompt"]],
            expected_output=[row["expected_output"]],
        )[0]
        held_out_predictions.append(
            {
                "index": index,
                "input": row["prompt"][-1]["content"],
                "expected_output": row["expected_output"],
                "completion": completion,
                "reward": held_out_reward,
            }
        )

held_out_rewards = [float(row["reward"]) for row in held_out_predictions]
for held_out_reward in held_out_rewards:
    assert math.isfinite(held_out_reward) and -1.0 <= held_out_reward <= 1.0

reward_values = []
for log_entry in trainer.state.log_history:
    if "reward" in log_entry:
        reward_values.append(float(log_entry["reward"]))

training_summary = {
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "gpu_name": torch.cuda.get_device_name(0),
    "mamba_fast_path_available": mamba_fast_path_available,
    "versions": {package: version(package) for package in PACKAGE_NAMES},
    "global_step": int(trainer.state.global_step),
    "training_loss": training_loss,
    "evaluation_ran": run_evaluation,
    "evaluation_metrics": numeric_validation_metrics,
    "held_out_reward_count": len(held_out_rewards),
    "held_out_reward_mean": (
        sum(held_out_rewards) / len(held_out_rewards) if held_out_rewards else None
    ),
    "held_out_reward_min": min(held_out_rewards) if held_out_rewards else None,
    "held_out_reward_max": max(held_out_rewards) if held_out_rewards else None,
    "held_out_perfect_rate": (
        sum(reward == 1.0 for reward in held_out_rewards) / len(held_out_rewards)
        if held_out_rewards
        else None
    ),
    "reward_log_count": len(reward_values),
    "reward_min": min(reward_values) if reward_values else None,
    "reward_max": max(reward_values) if reward_values else None,
}
result_path = os.getenv("GRPO_RESULT_PATH")
if result_path:
    result_file = Path(result_path)
    result_file.parent.mkdir(parents=True, exist_ok=True)
    result_file.write_text(
        json.dumps(training_summary, indent=2, sort_keys=True), encoding="utf-8"
    )
    predictions_file = result_file.with_name("held_out_predictions.jsonl")
    predictions_file.write_text(
        "".join(json.dumps(row, ensure_ascii=False) + "\n" for row in held_out_predictions),
        encoding="utf-8",
    )
training_summary

## 7. Save a merged inference checkpoint

In [ ]:
import shutil

from huggingface_hub import hf_hub_download

OUTPUT_DIR = Path(GRPO_OUTPUT_DIR) / "merged_model"
if os.getenv("GRPO_SAVE_MERGED", "1") == "1":
    model.save_pretrained_merged(
        str(OUTPUT_DIR),
        tokenizer,
        save_method="merged_16bit",
    )
    for filename in ("modeling_nemotron_h.py", "LICENSE", "__init__.py"):
        source = hf_hub_download(
            repo_id=MODEL_NAME, filename=filename, revision=MODEL_REVISION
        )
        shutil.copy2(source, OUTPUT_DIR / filename)
    for filename in (
        "config.json",
        "configuration_nemotron_h.py",
        "modeling_nemotron_h.py",
        "model.safetensors",
        "tokenizer.json",
    ):
        assert (OUTPUT_DIR / filename).is_file(), f"Merged checkpoint is missing {filename}"
    print(OUTPUT_DIR)
else:
    print("Skipping merged checkpoint because GRPO_SAVE_MERGED=0")